<a href="https://colab.research.google.com/github/rishabhsvats/deeplearning/blob/main/transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
device = "cuda" if torch.cuda.is_available() else "cpu"
device

# Pytorch Transfer Learning

What is transfer learning

Transfer learning involves taking the parameters of what one model has learned on another dataset and applying to our problem.

* Pretrained model = foundational model


In [ ]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)

In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
import torch
import torchvision

from torch import nn
from torchvision import transforms

# Try to get torchinfo, install it if it doesn't work
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

# Try to import the going_modular directory, download it from GitHub if it doesn't work
try:
    from going_modular.going_modular import data_setup, engine
except:
    # Get the going_modular scripts
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular.going_modular import data_setup, engine

## 1. Get Data
We need our pizza, steak, sushi data to build a transfer learning model on.

In [ ]:
import os
import zipfile

from pathlib import Path

import requests

#Setup data path
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"
# If the image folder doesn't exist, download it and prepare it...
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

    # Download pizza, steak, sushi data
    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping pizza, steak, sushi data...")
        zip_ref.extractall(image_path)

    # Remove .zip file
    os.remove(data_path / "pizza_steak_sushi.zip")

In [ ]:
# Setup directory path
train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

## 2. Create datasets and dataloaders

Now we have data and want to turn pytorch dataloaders.

To do this we can use `data_setup.py` and create_dataloaders() function

There 's one thing we have to think about when loading: how to **transform** it ?

ANd with `torchvision` 0.13+ there is two ways to do this:
1. Manually created transforms
2. Automatically created transforms - the transforms for your data are defined by the model you'd like to use.

IMportant point: When usign a pretrained model, its important that the data (including the custom data) that you pass through transformed in the same way that the data the model was trained on.

### 2.1 Creating a transform for `torchvision.models` (manual)

`torchvision.models` contains pretrained models (models ready for transfer learning) right within `torchvision`


All pre-trained models expect input images normalized in the same way, i.e. mini-batches of 3-channel RGB images of shape (3 x H x W), where H and W are expected to be at least 224.

The images have to be loaded in to a range of [0, 1] and then normalized using mean = [0.485, 0.456, 0.406] and std = [0.229, 0.224, 0.225].

In [ ]:
from torchvision import transforms
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
# Create a transforms pipeline manually (required for torchvision < 0.13)
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # 1. Reshape all images to 224x224 (though some models may require different sizes)
    transforms.ToTensor(), # 2. Turn image values to between 0 & 1
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # 3. A mean of [0.485, 0.456, 0.406] (across each colour channel)
                         std=[0.229, 0.224, 0.225]) # 4. A standard deviation of [0.229, 0.224, 0.225] (across each colour channel),
])

In [ ]:
from going_modular.going_modular import data_setup

# Create training and testing DataLoaders as well as get a list of class names
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=manual_transforms, # resize, convert images to between 0 & 1 and normalize them
                                                                               batch_size=32) # set mini-batch size to 32

train_dataloader, test_dataloader, class_names

### 2.2 Creating a transform for `torchvision.models` (auto creating)

In [ ]:
import torchvision
torchvision.__version__

In [ ]:
# Get a set of pretrained model weights

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
weights

In [ ]:
# Get the transforms used to create our pretrained weights

auto_transforms = weights.transforms()
auto_transforms

In [ ]:
#Create Dataloaders using automatic transforms.

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms, # resize, convert images to between 0 & 1 and normalize them
                                                                               batch_size=32) # set mini-batch size to 32

train_dataloader, test_dataloader, class_names

## 3. Getting a pretrained mod

The whole idea of transfer learning is to take an already well-performing model on a problem-space similar to yours and then customise it to your use case.

Since we're working on a computer vision problem (image classification with FoodVision Mini), we can find pretrained classification models in torchvision.models.

### 3.1 Which pretrained model should you use ?

Three things to consider:
1. Speed : how fast does it need to run ?
2. Size : how big is the model ?
3. Performance: how well does it go on your chosen problem.

Where does the model live?

Is it on device ? Or does it live on a server ?

Which model should we choose ?  


### 3.2 Setting up a pretrained model

Want to create pretrained EfficientNet_B0 model from

In [ ]:
# OLD method of creating a pretrained model (prior to torchvision v0.13)
#model = torchvision.models.efficientnet_b0(pretrained=True)

# New method of creating a pretrained model
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # .DEFAULT = best available weights for ImageNet
model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

### 3.3 Getting a summary of our model with torchinfo.summary()

In [ ]:
# print with torchinfo

from torchinfo import summary


summary(model=model,
        input_size=(1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"], # column in torchinfo table
        col_width=20,
        row_settings=["var_names"])

### 3.4 Freezing the base model and changing the output layer to suit our needs

With a feature extractor model. typically you will freeze the base layers of a pretrained model and update the output layers to suit your own problem.

In [ ]:
# Freeze all of the base layers in effnetb0
for param in model.parameters():
    param.requires_grad = False



In [ ]:
# update the classifier head of our model to suit or
from torch import nn

torch.manual_seed(42)
torch.cuda.manual_seed(42)
# Create a new classifier head
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, # feature vector coming in
              out_features=len(class_names),
              bias=True).to(device)
)
model.classifier

In [ ]:
summary(model=model,
        input_size=(1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"], # column in torchinfo table
        col_width=20,
        row_settings=["var_names"])

## 4. Train Model

In [ ]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Import train function

from going_modular.going_modular import engine

# Set the manual seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Start the timer

from timeit import default_timer as timer
start_time = timer()

# Setup training and save the results
results = engine.train(model=model,
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader,
                       optimizer=optimizer,
                       loss_fn=loss_fn,
                       epochs=10,
                       device=device)


# End the timer and print out how long it took

end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

## 5. Evaluate model by plotting loss curves

In [ ]:
try:
  from helper_functions import plot_loss_curves
except:
  print(f"Could not find helper functions.py, downloading...")
  with open("helper_functions.py", "wb") as f:
    import requests
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/refs/heads/main/helper_functions.py")
    f.write(request.content)
  from helper_functions import plot_loss_curves

plot_loss_curves(results)

## 6. Make predictions on images from the test set

Some things to keep in mind when making predictions/inference on test data/custom data

We have to make sure that our test/custom data is:
* Same shape - Images need to be same shape as model was trained on
* Same datatype - custom data should be in the same data type
* Same device - custom data/test data should be on the same device as the model.
* Same transform - if you have transformed your custom data, ideally you will transform the test data and custom data the same.

To do all of this automagically lets create a function called `pred_and_plot_image()`:

1. Take in a trained model, a list of class names, a filepath to a target image , an image size, a transform and a target device.
2. Open the image with `PIL.Image.Open()`
3. Create a transform if one doesnot exist.
4. Make sure the model is on the target device.
5. Turn the model to `model.eval()` mode to make sure it's ready for inference. (this will turn off things like `nn.Dropout()`)
6. Transform the target image and make sure its dimensionality is suited for the model.
7. Make a prediction on the image by passing to the model.
8. Convert the model's output logits to prediction probabilities using `torch.softmax()`
9. Convert model's prediction probabilities to prediction labels using `torch.argmax()`
10. Plot the image with `matplotlib` and set the title to the prediction label from step 9. and prediction probability from step 8.

In [ ]:
from typing import List, Tuple
from PIL import Image

from torchvision import transforms
def pred_and_plot_image(model: torch.nn.Module,
                        image_path: str,
                        class_names: List[str],
                        image_size: Tuple[int, int] = (224, 224),
                        transform: torchvision.transforms = None,
                        device: torch.device=device):
  #2. Open the image with PIL
  img = Image.open(image_path)

  #3. Create a transform if one does not exist
  if transform is None:
    image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
  else:
    image_transform = transform

  ### Predict on image ###

  #4. Make sure the model is on the target device
  model.to(device)

  #5. Turn on inference mode
  model.eval()
  with torch.inference_mode():
    # 6. Transform the image and add an extra batch dimension
    transformed_image = image_transform(img).unsqueeze(dim=0)

    # 7. Make a prediction on image with model
    target_image_pred = model(transformed_image.to(device))

  #8. Convert the model's output logits to pred probs
  target_image_pred_probs = torch.softmax(target_image_pred, dim=1)

  #9. Convert pred probs to pred labels
  target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

  #10. Plot the image using matplotlib
  plt.figure()
  plt.imshow(img)
  plt.title(f"Pred: {class_names[target_image_pred_label.cpu()]} | Prob: {target_image_pred_probs.max().cpu():.3f}")
  plt.axis(False)

In [ ]:
# Get a random list of image paths from the test set
import random
num_images_to_plot = 3
test_image_path_list = list(Path(test_dir).glob("*/*.jpg"))
test_image_path_sample = random.sample(population=test_image_path_list, k=num_images_to_plot)

# Make prediction and Plot the images
for image_path in test_image_path_sample:
  pred_and_plot_image(model=model,
                      image_path=image_path,
                      class_names=class_names,
                      image_size=(224,224))

## 6.1 Making predictions on a custom image

Let's make a prediction on custom image

In [ ]:
# Download the image
import requests

# Setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

if not custom_image_path.is_file():
  with open(custom_image_path, "wb") as f:
    custom_image_url = "https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/data/04-pizza-dad.jpeg"
    request = requests.get(custom_image_url)
    print(f"Download {custom_image_path}")
    f.write(request.content)
else:
  print(f"File already exists at {custom_image_path}")



In [ ]:
# Predict on custom image
pred_and_plot_image(model=model,
                    image_path=custom_image_path,
                    class_names=class_names)